# Week 2.4: RAG — Letting Your Twin Look Things Up
### Plain-English edition — no coding required

> **The pipeline: Load → Split → Embed → Retrieve → Answer**

A model only knows what it was trained on, plus whatever you put in front of it. It has never
read *your* notes, and you cannot paste all of them into every message (Week 2.2 showed why).

**RAG** — *Retrieval-Augmented Generation* — fixes that. Before your twin answers, it
**looks up** the few pieces of your notes that matter for this question, and answers from those.

This notebook puts the whole of Week 2 together into one working pipeline.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Load** — bring in the notes your twin should know
- **Split** — chop them into chunks (Week 2.1)
- **Embed** — turn every chunk into numbers (Week 2.3)
- **Retrieve** — find the chunks closest to a question
- **Augment** — build the exact message that gets sent
- **Answer** — let the twin reply, using only what was retrieved
- **Grounding** — what happens when the answer is not in the notes

**Steps 1–5 need no account.** Steps 6 and 7 call a real model through Groq, using the same
free `GROQ_API_KEY` secret as Week 1. Never paste the key into a box — use Colab's key icon.

---
## Part 1: Load — what should your twin know?

**Sami's notes** — seven short paragraphs about his life:

```
My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning.
```

Add facts of your own below — later steps will use them.

In [ ]:
#@title ▶ Step 1 — Load my notes { display-mode: "form" }
#@markdown Add up to two notes of your own (leave blank to skip):
extra_note_1 = "My favourite drink is spiced chai with ginger." #@param {type:"string"}
extra_note_2 = "" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

text = DEFAULT_NOTES
added = [t.strip() for t in (extra_note_1, extra_note_2) if t.strip()]
for t in added:
    text += "\n" + (t if t.endswith((".", "!", "?")) else t + ".")
globals()["_NOTES"] = text
paras = [p for p in text.split("\n") if p.strip()]
print("Loaded " + str(len(paras)) + " paragraphs, " + str(len(sentences_of(text))) + " sentences, "
      + str(len(text.split())) + " words, about " + str(tokens(text)) + " tokens.")
print("")
for i, p in enumerate(paras, 1):
    tag = "  (yours)" if i > len(DEFAULT_NOTES.split("\n")) else ""
    print(str(i).rjust(3) + ". " + p[:70] + ("..." if len(p) > 70 else "") + tag)
print("")
if added:
    success("Your " + str(len(added)) + " note(s) were added. The later steps will search them too.")
else:
    note("You did not add any notes of your own - that is fine, Sami's are enough.")
print("")
print("This is small enough to send whole. A real twin might have hundreds of pages,")
print("which is why the next four steps exist.")

---
## Part 2: Split — chop the notes into chunks

Each chunk becomes one searchable unit. **Too big** and every chunk is about everything, so
search gets vague. **Too small** and a single fact gets cut in half (you saw this in Week 2.1).

In [ ]:
#@title ▶ Step 2 — Chop my notes into chunks { display-mode: "form" }
#@markdown How many sentences per chunk?
sentences_per_chunk = 2 #@param {type:"slider", min:1, max:4, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

globals()["_PER"] = sentences_per_chunk
chunks = my_chunks()
print("Cutting into chunks of up to " + str(sentences_per_chunk) + " sentence(s), never across a paragraph:")
print("")
for i, c in enumerate(chunks, 1):
    print("chunk " + str(i).rjust(2) + "  (" + str(tokens(c)).rjust(2) + " tokens)  " + c[:62] + ("..." if len(c) > 62 else ""))
print("")
print(str(len(chunks)) + " chunks.")
if sentences_per_chunk == 1:
    note("One sentence per chunk: very precise, but 'My best 5km time is 23 minutes' "
         "is now separated from 'I go running' - that link is lost.")
elif sentences_per_chunk >= 4:
    note("Big chunks: each one now covers several topics, so retrieval will drag in "
         "text that has nothing to do with the question.")
else:
    success("A sensible middle size - each chunk is about one thing.")

---
## Part 3: Embed — give every chunk a position on the map

Exactly what you did in Week 2.3: each chunk becomes a list of numbers, so that chunks about
similar things land near each other. This list of numbers, one per chunk, is your **index**.

In [ ]:
#@title ▶ Step 3 — Turn every chunk into numbers { display-mode: "form" }
#@markdown How many of each chunk's numbers should I print?
numbers_to_show = 5 #@param {type:"slider", min:2, max:10, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

chunks = my_chunks()
index = build_index(chunks)
print("Built a map from " + str(len(index["vocab"])) + " different words, and placed "
      + str(len(chunks)) + " chunks on it.")
print("")
missing = 0
for i, (c, v) in enumerate(zip(chunks, index["vecs"]), 1):
    if v is None:
        missing += 1
        print("chunk " + str(i).rjust(2) + "  [ no position - no known words ]")
    else:
        nums = "  ".join(format(x, "+.2f") for x in v[:numbers_to_show])
        print("chunk " + str(i).rjust(2) + "  [ " + nums + "  ... " + str(len(v)) + " numbers ]")
print("")
if missing:
    problem(str(missing) + " chunk(s) could not be placed, so they can never be found.")
else:
    success("Every chunk has a position. The index is ready to search.")
print("")
print("In a real product this list is stored in a 'vector database', so it is built")
print("once and searched many times - not rebuilt for every question.")

---
## Part 4: Retrieve — find the chunks that answer the question

The question gets embedded the same way. Then we pick the chunks sitting closest to it.
Two dials matter: **how many** chunks to take, and **how close** a chunk must be to count at all.

In [ ]:
#@title ▶ Step 4 — Find the chunks that answer my question { display-mode: "form" }
question = "How fast can you run 5km?" #@param {type:"string"}
#@markdown How many chunks to retrieve:
top_k = 2 #@param {type:"slider", min:1, max:5, step:1}
#@markdown Ignore chunks scoring below:
min_score = 0.3 #@param {type:"slider", min:0.0, max:0.9, step:0.05}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

if not question.strip():
    problem("Please type a question, then press play again.")
else:
    index = build_index(my_chunks())
    scored, picked = retrieve(index, question, top_k, min_score)
    print("Question: " + question)
    print("")
    if scored is None:
        problem("None of the words in that question appear anywhere in the notes or the map.")
        note("So nothing can be retrieved. Rephrase using everyday words - run, eat, work, sleep, music...")
    else:
        print("Every chunk, scored against the question:")
        print("")
        for s, i, c in scored:
            mark = " <-- retrieved" if (s, i, c) in picked else ""
            print("   " + simbar(s, 24) + " " + f2(s) + "  " + c[:40] + ("..." if len(c) > 40 else "") + mark)
        print("")
        if not picked:
            problem("Nothing scored above " + format(min_score, ".2f") + ", so nothing was retrieved.")
            note("Either the answer is not in the notes, or the bar is set too high. Try lowering 'min score'.")
        else:
            success("Retrieved " + str(len(picked)) + " chunk(s). Only these will be shown to the model.")
            if len(picked) < top_k:
                note("You asked for " + str(top_k) + ", but only " + str(len(picked))
                     + " cleared the minimum score. That is the filter doing its job.")

---
## Part 5: Augment — see exactly what gets sent

This is the **A** in RAG, and the part most people never look at. The retrieved chunks are
pasted into the message, together with a firm instruction: *answer from these, and only these.*

In [ ]:
#@title ▶ Step 5 — Show me the exact message that gets sent { display-mode: "form" }
question = "How fast can you run 5km?" #@param {type:"string"}
top_k = 2 #@param {type:"slider", min:1, max:5, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

if not question.strip():
    problem("Please type a question, then press play again.")
else:
    index = build_index(my_chunks())
    scored, picked = retrieve(index, question, top_k, 0.3)
    msgs = build_prompt(question, picked)
    print("=" * 70)
    print("INSTRUCTION (sent every time):")
    show_text("", msgs[0]["content"])
    print("")
    print("MESSAGE:")
    show_text("", msgs[1]["content"])
    print("=" * 70)
    sent = tokens(msgs[0]["content"] + msgs[1]["content"])
    whole = tokens(my_notes())
    print("")
    print("Size of this message:      about " + str(sent) + " tokens")
    print("Size of ALL the notes:     about " + str(whole) + " tokens")
    print("")
    if picked:
        success("The model sees " + str(len(picked)) + " of " + str(len(index["chunks"]))
                + " chunks - only what this question needs.")
    else:
        problem("Nothing was retrieved, so the model is told the notes have nothing relevant.")
    print("")
    print("With seven paragraphs the saving is small. With seven hundred pages, this is")
    print("the difference between a message that fits and one that cannot be sent at all.")

---
## Part 6: Answer — let your twin reply

Now a real model answers. Tick **compare** to ask the same question twice: once with the
retrieved notes, once with no notes at all. The difference is the whole point of RAG.

### Before you press play: connect your Groq API key

Steps 6 and 7 talk to a real model, so they need a **free Groq API key**. If you already
added one in Week 1, you are done — Colab keeps it for you. Otherwise, it takes two minutes:

1. Go to **[console.groq.com](https://console.groq.com)** and sign up (free, no card needed)
2. Open **API Keys** → **Create API Key**, give it any name, and **copy** the key
   (it starts with `gsk_` — you will only be shown it once)
3. Back here in Colab, click the **🔑 key icon** in the left sidebar
4. Click **Add new secret**
5. In **Name**, type exactly **`GROQ_API_KEY`** — capitals and underscores matter
6. In **Value**, paste your key
7. Switch **Notebook access** on
8. Press play on Step 6

> **Never paste your key into a step, a box, or a chat.** Anyone who sees your notebook —
> or a screenshot, or a video of your screen — would have your key and your bill.

**If something goes wrong,** Step 6 tells you in red:
- *"I could not find your GROQ_API_KEY"* — check the name is spelled exactly, and that
  **Notebook access** is switched on
- *"The server rejected the key ending in ..."* — the key was mistyped or deleted; make a
  new one at console.groq.com and replace the value in the 🔑 panel
- Changed your key? Just press play again — the notebook re-reads it every time, so there is
  no need to restart.

In [ ]:
#@title ▶ Step 6 — Let my twin answer from my notes { display-mode: "form" }
question = "How fast can you run 5km?" #@param {type:"string"}
top_k = 2 #@param {type:"slider", min:1, max:5, step:1}
#@markdown Also ask WITHOUT the notes, to see the difference?
compare_without_notes = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.6-27b", "qwen/qwen3.8-27b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    globals().pop("_BADKEY", None); globals().pop("_LIST_ERR", None)
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        else:
            globals()["_LIST_ERR"] = str(e)
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def probe_model(c):
    # the model list was unavailable, so ask likely models directly; keep the first that answers
    tried = []
    for m in PREFERRED:
        try:
            c.chat.completions.create(model=m, messages=[{"role": "user", "content": "hi"}], max_tokens=16)
            return m, tried
        except Exception as e:
            t = str(e)
            if "401" in t or "invalid_api_key" in t:
                globals()["_BADKEY"] = True
                return None, tried
            tried.append((m, t[:90]))
    return None, tried
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    if globals().get("_BADKEY"): return None
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    globals()["_PROBE_TRIED"] = []
    if not pick:
        pick, tried = probe_model(c)
        globals()["_PROBE_TRIED"] = tried
    if pick: globals()["_MODEL"] = pick
    return pick
def explain_no_model():
    problem("I could not get any model to answer with this key.")
    err = globals().get("_LIST_ERR")
    if err: note("Asking for the list of models failed with: " + err[:170])
    tried = globals().get("_PROBE_TRIED") or []
    if tried:
        note("Models I tried directly, and what the server said:")
        for m, why in tried[:4]: note("   " + m + "  ->  " + why)
    note("At console.groq.com/keys, check whether this key is limited to certain models or projects.")
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        explain_no_model(); return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

if not question.strip():
    problem("Please type a question, then press play again.")
else:
    index = build_index(my_chunks())
    scored, picked = retrieve(index, question, top_k, 0.3)
    print("Question: " + question)
    print("")
    if picked:
        print("Retrieved " + str(len(picked)) + " chunk(s):")
        for s, i, c in picked:
            print("   [" + f2(s) + "] " + c[:66] + ("..." if len(c) > 66 else ""))
    else:
        note("Nothing relevant was retrieved - the twin is told so.")
    print("")
    with_notes = llm(build_prompt(question, picked), temperature=0.2, max_tokens=300)
    if with_notes:
        show("WITH my notes", with_notes)
        if compare_without_notes:
            print("")
            bare = [{"role": "system", "content": "You are Sami's digital twin. Answer in the first person, briefly."},
                    {"role": "user", "content": question}]
            without = llm(bare, temperature=0.2, max_tokens=300)
            if without:
                show("WITHOUT any notes", without)
                print("")
                print("Look at the second answer. With nothing to look up, the model either")
                print("admits it does not know, or - worse - makes up something plausible.")
                print("Both sound equally confident. Only the first is grounded in a fact.")
        print("")
        success("That is RAG: Load -> Split -> Embed -> Retrieve -> Answer.")

---
## Part 7: Grounding — ask about something that is not there

A good RAG system knows when it does **not** know. Ask about something the notes never
mention and check that your twin says so, instead of inventing an answer.

In [ ]:
#@title ▶ Step 7 — Ask about something that is not in my notes { display-mode: "form" }
question = "Do you have a pet dog?" #@param {type:"string"}
min_score = 0.3 #@param {type:"slider", min:0.0, max:0.9, step:0.05}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]
CORPUS = [
 "i love running every morning before work it keeps me healthy",
 "running and swimming are my favourite sport exercise",
 "exercise keeps the body healthy strong and fit",
 "football match team play sport running goal",
 "swimming running exercise healthy body fit morning",
 "morning run before work then coffee and breakfast",
 "i eat rice and beans for lunch it is tasty food",
 "food rice beans chicken lunch tasty filling meal",
 "cooking food eating lunch with family tasty meal",
 "rice chicken food dinner eating tasty evening meal",
 "coffee breakfast morning food drink tasty",
 "i write code in python on my laptop computer",
 "python code laptop computer software program write",
 "programming code software laptop computer write evening",
 "computer laptop software program code python work",
 "work office project deadline computer email",
 "family friends happy love together joy laughter",
 "happy friends family laughter joy together weekend",
 "sad tired alone unhappy feeling low",
 "friends family together love happy weekend",
 "school study books learning exams reading",
 "study books school learn exams reading notes",
 "school exams study learning books teacher",
 "music song dance listen radio evening",
 "song music dance happy listen weekend",
 "sleep rest night bed tired early",
 "night sleep bed rest early morning tired",
 "accra ghana city home street market",
 "ghana accra market city home people",
]
THEMES = {
 "sport":    ["running","swimming","exercise","sport","football","fit","healthy","body","match","team","play"],
 "food":     ["rice","beans","food","chicken","lunch","tasty","eat","eating","meal","cooking","dinner","coffee","breakfast"],
 "computer": ["code","python","laptop","computer","software","program","programming","write","work","office"],
 "people":   ["family","friends","happy","love","together","joy","laughter","weekend","sad","tired"],
 "school":   ["school","study","books","learning","exams","reading","learn","notes","teacher"],
}
STOP = {"the","a","an","and","my","i","is","are","was","in","for","with","of","to",
        "me","it","on","that","this","am","then","before","every"}
def content_words(text):
    return [w for w in words_of(text) if w not in STOP]
def build_space(dim=24, window=3):
    T = [content_words(c) for c in CORPUS]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); rows = M.sum(1, keepdims=True); cols = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (rows * cols + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    V = U[:, :dim] * S[:dim]
    V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    flat = U[:, :2] * S[:2]
    return vocab, idx, V, M, flat
def space():
    if "_SPACE" not in globals(): globals()["_SPACE"] = build_space()
    return globals()["_SPACE"]
def vec(word):
    vocab, idx, V, M, flat = space()
    return V[idx[word]] if word in idx else None
def near(word, k=5):
    vocab, idx, V, M, flat = space()
    sims = V.dot(V[idx[word]]); order = np.argsort(-sims)
    return [(vocab[i], float(sims[i])) for i in order if vocab[i] != word][:k]
def sim(a, b):
    va, vb = vec(a), vec(b)
    return None if va is None or vb is None else float(va.dot(vb))
def theme_of(word):
    for t, ws in THEMES.items():
        if word in ws: return t
    return "other"
def sentence_vec(text):
    vocab, idx, V, M, flat = space()
    vs = [V[idx[w]] for w in content_words(text) if w in idx]
    if not vs: return None
    v = np.mean(vs, axis=0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else None
def simbar(score, width=34):
    filled = max(0, min(width, int(round(width * max(0.0, score)))))
    return "[" + "#" * filled + "." * (width - filled) + "]"
def verdict(score):
    if score >= 0.7: return "very close in meaning"
    if score >= 0.4: return "related"
    if score >= 0.15: return "loosely connected"
    return "nothing to do with each other"
def f2(x):
    return format(0.0 if abs(x) < 0.005 else x, ".2f")
import re
DEFAULT_NOTES = """My name is Sami and I live in Mombasa, Kenya. I grew up in the Old Town near the market with my family.
I work as a pharmacist at a hospital pharmacy. I check prescriptions and explain to patients how to take their medicine safely.
I go running every morning before work, along Nyali beach. My best 5km time is 23 minutes. I am training for a 10km race in March.
For lunch I usually eat rice and beans. On Fridays my family cooks biryani with chicken for dinner.
In the evenings I am studying for a clinical pharmacy exam in June. I read books and notes on the matatu home.
At the weekend I play football with friends and listen to taarab music.
I try to sleep early, by ten at night, because the pharmacy opens at seven in the morning."""
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
def my_notes():
    return globals().get("_NOTES", DEFAULT_NOTES)
def chunk_notes(text, per=2):
    out = []
    for para in [p for p in text.split("\n") if p.strip()]:
        s = sentences_of(para)
        for i in range(0, len(s), max(1, per)):
            out.append(" ".join(s[i:i + max(1, per)]))
    return out
def my_chunks():
    return chunk_notes(my_notes(), globals().get("_PER", 2))
def build_index(chunks, dim=32, window=3):
    T = [content_words(c) for c in CORPUS + chunks]
    vocab = sorted({w for t in T for w in t}); idx = {w: i for i, w in enumerate(vocab)}
    M = np.zeros((len(vocab), len(vocab)))
    for t in T:
        for i, w in enumerate(t):
            for j in range(max(0, i - window), min(len(t), i + window + 1)):
                if i != j: M[idx[w], idx[t[j]]] += 1.0
    tot = M.sum(); r = M.sum(1, keepdims=True); c = M.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P = np.log((M * tot) / (r * c + 1e-12) + 1e-12)
    P[~np.isfinite(P)] = 0; P[P < 0] = 0
    U, S, Vt = np.linalg.svd(P, full_matrices=False)
    d = min(dim, U.shape[1])
    V = U[:, :d] * S[:d]; V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
    def embed(text):
        vs = [V[idx[w]] for w in content_words(text) if w in idx]
        if not vs: return None
        v = np.mean(vs, axis=0); nrm = np.linalg.norm(v)
        return v / nrm if nrm > 0 else None
    vecs = [embed(c) for c in chunks]
    return {"chunks": chunks, "vecs": vecs, "embed": embed, "vocab": vocab}
def retrieve(index, question, k=2, min_score=0.3):
    qv = index["embed"](question)
    if qv is None: return None, []
    scored = []
    for i, (c, v) in enumerate(zip(index["chunks"], index["vecs"])):
        s = float(qv.dot(v)) if v is not None else 0.0
        scored.append((s, i, c))
    scored.sort(reverse=True)
    return scored, [x for x in scored[:k] if x[0] >= min_score]
def build_prompt(question, picked):
    if picked:
        notes = "\n".join("- " + c for s, i, c in picked)
    else:
        notes = "(nothing relevant was found in the notes)"
    system = ("You are Sami's digital twin. Answer in the first person, briefly and warmly. "
              "Use ONLY the facts in the NOTES. If the notes do not contain the answer, "
              "say plainly that it is not in your notes. Never invent facts.")
    user = "NOTES:\n" + notes + "\n\nQUESTION: " + question
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]
def tokens(text):
    return max(1, round(len(text) / 4))
def show_text(label, text, width=76):
    if label: print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)

PREFERRED = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant",
             "llama3-70b-8192", "llama3-8b-8192", "mixtral-8x7b-32768", "gemma2-9b-it",
             "openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.6-27b", "qwen/qwen3.8-27b"]
SKIP = ("whisper", "tts", "guard", "orpheus", "embed")
def say_rejected(k):
    problem("The server rejected the key ending in ..." + (k[-4:] if k else "????") + ".")
    note("Open console.groq.com/keys and compare the last 4 characters with your active key.")
    note("If they differ, update the GROQ_API_KEY secret, then press play again.")
    note("If the key was revoked or deleted, create a new one and save that instead.")
def usable_models(c):
    globals().pop("_BADKEY", None); globals().pop("_LIST_ERR", None)
    try:
        ids = [m.id for m in c.models.list().data]
    except Exception as e:
        if "401" in str(e) or "invalid_api_key" in str(e):
            globals()["_BADKEY"] = True
        else:
            globals()["_LIST_ERR"] = str(e)
        return []
    return [i for i in ids if not any(s in i.lower() for s in SKIP)]
def probe_model(c):
    # the model list was unavailable, so ask likely models directly; keep the first that answers
    tried = []
    for m in PREFERRED:
        try:
            c.chat.completions.create(model=m, messages=[{"role": "user", "content": "hi"}], max_tokens=16)
            return m, tried
        except Exception as e:
            t = str(e)
            if "401" in t or "invalid_api_key" in t:
                globals()["_BADKEY"] = True
                return None, tried
            tried.append((m, t[:90]))
    return None, tried
def choose_model(c):
    if globals().get("_MODEL"): return globals()["_MODEL"]
    avail = usable_models(c)
    if globals().get("_BADKEY"): return None
    pick = next((m for m in PREFERRED if m in avail), None) or (avail[0] if avail else None)
    globals()["_PROBE_TRIED"] = []
    if not pick:
        pick, tried = probe_model(c)
        globals()["_PROBE_TRIED"] = tried
    if pick: globals()["_MODEL"] = pick
    return pick
def explain_no_model():
    problem("I could not get any model to answer with this key.")
    err = globals().get("_LIST_ERR")
    if err: note("Asking for the list of models failed with: " + err[:170])
    tried = globals().get("_PROBE_TRIED") or []
    if tried:
        note("Models I tried directly, and what the server said:")
        for m, why in tried[:4]: note("   " + m + "  ->  " + why)
    note("At console.groq.com/keys, check whether this key is limited to certain models or projects.")
def _read_key():
    k = None
    try:
        from google.colab import userdata
        k = userdata.get("GROQ_API_KEY")
    except Exception:
        import os as _o; k = _o.environ.get("GROQ_API_KEY")
    return k.strip() if k and k.strip() else None
def forget_connection():
    for name in ("_CLIENT", "_KEY", "_MODEL"): globals().pop(name, None)
def _client():
    k = _read_key()
    if k and globals().get("_CLIENT") and globals().get("_KEY") == k:
        return globals()["_CLIENT"], k
    forget_connection()      # new or changed key: never reuse a connection made with the old one
    try:
        from groq import Groq
    except Exception:
        import subprocess, sys as _s
        subprocess.run([_s.executable, "-m", "pip", "install", "-q", "groq"], capture_output=True)
        try:
            from groq import Groq
        except Exception:
            problem("Could not install the groq library."); return None, None
    if not k:
        problem("I could not find your GROQ_API_KEY.")
        note("One-time setup, about two minutes:")
        note("   1. Get a free key at console.groq.com (no card needed)")
        note("   2. In Colab, click the key icon in the left sidebar")
        note("   3. Add a secret named exactly  GROQ_API_KEY  and paste your key in")
        note("   4. Switch ON 'Notebook access', then press play here again")
        note("Never paste your key into a box in this notebook - anyone you share it with gets it.")
        return None, None
    globals()["_CLIENT"] = Groq(api_key=k); globals()["_KEY"] = k
    return globals()["_CLIENT"], k
def mask(k): return k[:4] + "..." + k[-4:] if k and len(k) > 10 else "????"
def llm(messages, temperature=0.3, max_tokens=600, quiet=False, _retry=False):
    c, k = _client()
    if c is None: return None
    model = choose_model(c)
    if not model and globals().pop("_BADKEY", False):
        forget_connection(); say_rejected(k); return None
    if not model:
        explain_no_model(); return None
    try:
        r = c.chat.completions.create(model=model, messages=messages,
                                      temperature=temperature, max_tokens=max_tokens)
    except Exception as e:
        msg = str(e)
        if "401" in msg or "invalid_api_key" in msg:
            forget_connection(); say_rejected(k)
        elif "429" in msg:
            problem("Too many requests. Wait a few seconds and press play again.")
        else:
            problem("The request failed: " + msg[:110])
        return None
    msg = r.choices[0].message
    out = (getattr(msg, "content", None) or "").strip()
    if not quiet and getattr(r, "usage", None):
        globals()["_LAST_USAGE"] = r.usage
    if not out and not _retry:
        # reasoning models spend their budget thinking before answering: give them more room
        return llm(messages, temperature=temperature, max_tokens=max(1200, max_tokens * 4),
                   quiet=quiet, _retry=True)
    if not out:
        # last resort: the model only produced its thinking, so show that rather than nothing
        think = (getattr(msg, "reasoning", None) or "").strip()
        if think:
            note("This model thinks before answering, and ran out of room for the final answer.")
            note("Showing its working instead:")
            out = think
    if not out:
        problem("The model sent back an empty reply.")
        note("Some models think before answering and ran out of room. Try a shorter question.")
        return None
    return out
def show(title, text, width=74):
    import textwrap
    print("+" + "-" * width + "+")
    print("| " + title.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
    for para in (text or "").split(chr(10)):
        if not para.strip():
            print("| " + " " * (width - 2) + " |"); continue
        for line in textwrap.wrap(para, width - 4) or [""]:
            print("| " + line.ljust(width - 2) + " |")
    print("+" + "-" * width + "+")
def twin_rules():
    return globals().get("_SYSTEM",
        "You are a digital twin of Ama, a 17-year-old student from Accra who loves football. "
        "Answer briefly and warmly. Never claim to be a real human.")

if not question.strip():
    problem("Please type a question, then press play again.")
else:
    index = build_index(my_chunks())
    scored, picked = retrieve(index, question, 2, min_score)
    print("Question: " + question)
    print("")
    if scored is None:
        print("None of the question's words appear anywhere in the notes.")
    elif picked:
        print("Retrieved:")
        for s, i, c in picked:
            print("   [" + f2(s) + "] " + c[:66] + ("..." if len(c) > 66 else ""))
        note("Something was retrieved. If it does not really answer the question, "
             "the twin should still say so.")
    else:
        print("Nothing in the notes scored above " + format(min_score, ".2f") + ".")
    print("")
    reply = llm(build_prompt(question, picked), temperature=0.0, max_tokens=200)
    if reply:
        show("Twin's reply", reply)
        print("")
        low = reply.lower()
        honest = any(p in low for p in ("not in my notes", "don't have", "do not have", "no information",
                                         "not mentioned", "doesn't say", "does not say", "not sure",
                                         "don't know", "do not know", "isn't in", "is not in", "no mention"))
        if honest:
            success("Grounded: it said the answer is not in the notes instead of inventing one.")
        else:
            problem("Check that reply carefully - did it make something up?")
            note("If so, the instruction in Step 5 needs to be stricter. That is a real "
                 "bug in real products, and Week 3 is about catching it.")

---
## What you just built

| Stage | Step | What happened | Where you learned it |
|---|---|---|---|
| **Load** | 1 | Brought in the notes | — |
| **Split** | 2 | Chopped them into chunks | Week 2.1 |
| **Embed** | 3 | Placed every chunk on a map | Week 2.3 |
| **Retrieve** | 4 | Found the chunks nearest the question | Week 2.3 |
| **Augment** | 5 | Pasted them into the message | Week 2.2 |
| **Answer** | 6 | Let the model reply from them | Week 1 |
| **Ground** | 7 | Checked it admits what it does not know | — |

### Three things that go wrong in real RAG systems
1. **Bad chunks** — the fact was split in half, so no single chunk holds it (Step 2).
2. **Bad retrieval** — the right chunk exists but scores too low to be picked (Step 4).
3. **Bad grounding** — nothing relevant was retrieved, and the model answers anyway (Step 7).

When your twin gives a wrong answer, check them in that order. Most of the time it is not
the model — it is what you handed it.

### Want to see the code?
Click **Show code** on any step.